# Step 4: mixed-precision config_groups（课程高阶核心）

**目标**：掌握 `llmcompressor` 0.7.0+ 的 mixed-precision——在一次 `oneshot` 里用**多个 `QuantizationModifier` 实例**（不同 `targets` + `scheme`），让 llmcompressor 自动合并成 compressed-tensors 的 `config_groups`：敏感层配更高比特（如 W8A16）、其余层用低比特（如 W8A8）。这是「最少回退 → 最大精度」Pareto 优化的工程基础——比粗粒度 FP16 全回退**省显存**又**保精度**。vLLM 0.10.1+ 原生可跑 mixed-precision 模型。

**对应 OUTLINE 课时**：3.4 细粒度混合精度回退 mixed-precision config_groups（~55 分钟，课程高阶核心）。


## 学完应能讲清（学完本节应能口头回答）
1. mixed-precision 比「敏感层 ignore 全回退 FP16」**好在哪**？（不浪费——敏感层用 W8A16 而非 FP16，省显存又保精度）
2. 怎么用**多个 `QuantizationModifier`** 表达「A 类层 W8A8、B 类层 W8A16」？它们怎么合并成 `config_groups`？
3. 为什么 mixed-precision **不能手动改 config.json**？（漏 `observer`/`observer_kwargs` 字段会让 vLLM 加载失败）
4. `targets` 接受正则吗？`targets=["re:.*down_proj"]` 和 `ignore=["re:.*down_proj"]` 一个指定量化的层、一个排除的层，区别在哪？
5. OUTLINE 说 vLLM 0.10.1+ 原生可跑 mixed-precision——这为什么是 mixed-precision 能落地的关键（否则要自己反量化回 FP16）？


In [ ]:
%%capture
import pathlib, os, re
import torch
import torch.nn as nn
import ipytest
ipytest.autoconfig()
from transformers import Qwen2Config, Qwen2ForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor.modifiers.gptq import GPTQModifier


In [ ]:
# Setup cell（双 env：模块根 = 含 scripts/ + steps/ 的 course/m3-tuning-eval/）。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT      = _find_module_root(pathlib.Path.cwd())
MODEL_DIR        = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"
TINY_MODEL_DIR   = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"
OUT_ROOT         = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT =", MODULE_ROOT, "| 0.5B @", TINY_MODEL_DIR.exists())


## 原理：mixed-precision = 多 modifier 合并 config_groups

OUTLINE 3.4 核心：

- llmcompressor 0.7.0+ 支持一次 `oneshot` 里传**多个 `QuantizationModifier`**，每个有自己的 `targets` + `scheme`。llmcompressor 自动把它们的并集整理成 compressed-tensors 的 `config_groups`：
  - `group_0`：低比特 targets（多数 attention/MLP 层，如 W8A8）
  - `group_1`：高比特 targets（敏感的 down_proj 等，如 W8A16）
- **示例结构**（产物 `config.json` 的 `quantization_config`）：
  ```json
  "config_groups": {
    "group_0": {"targets": ["Linear"], "input_activations": {...num_bits:8}, "weights": {...num_bits:8}},
    "group_1": {"targets": ["re:.*down_proj"], "weights": {...num_bits:16}, "input_activations": null}
  }
  ```

**关键易错点**（OUTLINE）：
- mixed-precision **不是手动改 config.json**——应通过多个 modifier 让 llmcompressor 生成正确结构。手动改极易漏 `observer`/`observer_kwargs` 字段，导致 vLLM 加载失败。
- `targets` 既接受类名（`"Linear"`）也接受正则（`["re:.*down_proj"]`）；`ignore` 同理。
- W8A16：权重 8→16 bit（实际是 weight-only 高精度，激活不量化）——给敏感层「留半精度权重」比「全回退 FP16」省。

**为什么 vLLM 原生支持 mixed-precision 是它落地的关键**（把第 5 题的 why 讲透，不止复述「否则要反量化回 FP16」）：

mixed-precision 的产物 `config.json` 里，**不同层的 `num_bits` 不同**（如 group_0 是 W8A8、group_1 是 W8A16）。推理引擎加载时必须**按组分别处理**：读到 group_0 的层按 INT8 反量化、读到 group_1 的层按 FP16 加载——这就是「原生支持 mixed config」。

如果引擎**不认 mixed config**（旧版 vLLM < 0.10.1，或只懂「整模型一个 scheme」的引擎），它没法按组分别处理，只剩两条烂路：
1. **整模型当 FP16 加载**：把敏感层也当 FP16，那 s4 给敏感层配的 W8A16 完全没生效——mixed-precision「敏感层用 W8A16 既省显存又保精度」的优势**全部作废**，退化成 s3 的「敏感层全回退 FP16」，等于 s4 白做。
2. **手动反量化回 FP16 再喂给引擎**：你自己写代码把 INT8 层反量化成 float、敏感层也保持 float，整个模型变成纯 FP16 推理——显存占用回到 FP16 水平，**省显存的优势同样作废**。

所以 OUTLINE 说「vLLM 0.10.1+ 原生可跑 mixed-precision」是 s4 能落地的**前提**：引擎能按 `config_groups` 的 `num_bits` 分别加载不同精度的层，mixed-precision 的省显存 + 保精度才同时成立。没有这个原生支持，mixed config 产物就成了「理论上漂亮、加载就塌」的摆设——这也是为什么 s4 的 L3 产物验证后，部署阶段必须确认 vLLM 版本 ≥ 0.10.1。

### 端到端：mixed-precision 在 layer fallback 全流程的位置

s1 找敏感层 → s2 经验法则 → s3 学了 ignore（**二值**：量化 or 全回退 FP16）→ **s4 升级成 mixed-precision**（**多值**：敏感层不一定要回退到 FP16，可配 W8A16 这种中间档）。s4 是 s3 的精细化：把「全回退」拆成「部分回退」，用更细的粒度逼近 Pareto 前沿。s5 的 Pareto 调优会混合用 ignore（全回退）和 mixed-precision（部分回退）两种手段。


## 亲手摸一摸：两个 modifier 怎么变 config_groups

构造两个 `QuantizationModifier`（一个 W8A8 全 Linear、一个 W8A16 只 down_proj），看它们的 `targets`/`scheme`/`ignore` 长什么样——这就是 config_groups 的「原料」。


In [ ]:
## 摸一摸：mixed-precision 两个 modifier 的字段
m_low  = QuantizationModifier(targets="Linear", scheme="W8A8", ignore=["re:.*down_proj", "lm_head"])
m_high = QuantizationModifier(targets=["re:.*down_proj"], scheme="W8A16", ignore=["lm_head"])
print("低比特组（多数层 W8A8）：")
print(f"  targets={m_low.targets}  scheme={m_low.scheme}  ignore={m_low.ignore}")
print("高比特组（敏感层 W8A16）：")
print(f"  targets={m_high.targets}  scheme={m_high.scheme}  ignore={m_high.ignore}")
print("\n这两个 modifier 组成 recipe list，oneshot 会合并成 config_groups（group_0=W8A8, group_1=W8A16）")
print("注意：低比特组必须 ignore 掉 down_proj，否则两组 targets 重叠（先 W8A8 再 W8A16，后者覆盖）")


## 本步填空

1. **`build_mixed_precision_recipe(sensitive_layers, low_scheme, high_scheme)`** —— 构造 mixed-precision recipe（list of 2 个 QuantizationModifier）：低比特组 targets=所有 Linear 但 ignore 敏感层+lm_head；高比特组 targets=敏感层（正则）配高 scheme。**为什么这么设计（填前先想）**：两组 targets 不能重叠——低比特组必须把敏感层 ignore 掉，否则同一层被两个 modifier 抢，行为未定义。recipe 是 modifier 的 list，oneshot 按顺序应用、最后合并 config_groups。
2. **`is_mixed_precision(quantization_config)`**（判断型）—— 从产物判断这是否是合法 mixed-precision：config_groups 至少 2 组、且各组 weights num_bits 不全相同。**为什么这么设计**：mixed-precision 的「证据」就是 config_groups 里有多组不同比特——一键裁决产物确实是 mixed（而非单档被误当 mixed），也是 L2/L3 验证产物正确的判据。


In [ ]:
def build_mixed_precision_recipe(sensitive_layers, low_scheme="W8A8", high_scheme="W8A16"):
    """返回 mixed-precision recipe（list of 2 个 QuantizationModifier）：
      - 低比特组：targets="Linear"，scheme=low_scheme，ignore = [敏感层正则..., "lm_head"]。
      - 高比特组：targets=[敏感层正则...]，scheme=high_scheme，ignore=["lm_head"]。

    为什么这么设计（填前先想）：
    - 两组 targets 必须不重叠：低比特组把敏感层 ignore 掉，高比特组专门 targets 敏感层。
    - 敏感层是一组层类后缀（如 ['down_proj']），用 s3 的 build_regex_ignore 思路转成一条
      're:.*down_proj|...' 正则，分别作为低组的 ignore 项和高组的 targets 项（复用同一正则）。
    - lm_head 两组都 ignore（铁律，s2）。
    """
    # TODO: 1) 用 s3 的正则思路把 sensitive_layers（层类后缀列表）转成一条合并正则
    #          sensitive_re，例如 body='.*down_proj' -> sensitive_re='re:.*down_proj'。
    #       2) 低比特组 m_low = QuantizationModifier(targets="Linear", scheme=low_scheme,
    #          ignore=[sensitive_re, "lm_head"])。
    #       3) 高比特组 m_high = QuantizationModifier(targets=[sensitive_re], scheme=high_scheme,
    #          ignore=["lm_head"])。
    #       4) 返回 [m_low, m_high]。
    raise NotImplementedError


In [ ]:
def is_mixed_precision(quantization_config):
    """判断型：产物是否是合法 mixed-precision。
    合法条件：config_groups 至少 2 组，且各组 weights num_bits 不全相同。
    返回 True/False。

    为什么这么设计（填前先想）：mixed-precision 的证据就是 config_groups 里有≥2 组、
    且比特数不同。一键裁决，防止单档被误当 mixed；也是 L3 验证产物正确的判据。
    """
    # TODO: 1) groups = quantization_config.get('config_groups', {})。
    #       2) 若 len(groups) < 2 返回 False。
    #       3) 收集每组 weights num_bits，若集合大小 < 2 返回 False（比特全相同不算 mixed）。
    #       4) 否则 True。
    raise NotImplementedError


In [ ]:
def test_build_mixed_precision_two_modifiers():
    recipe = build_mixed_precision_recipe(["down_proj"])
    assert isinstance(recipe, list) and len(recipe) == 2
    m_low, m_high = recipe
    assert isinstance(m_low, QuantizationModifier) and isinstance(m_high, QuantizationModifier)

def test_build_mixed_precision_schemes_differ():
    m_low, m_high = build_mixed_precision_recipe(["down_proj"])
    assert m_low.scheme != m_high.scheme, "两组 scheme 必须不同才算 mixed"
    assert "W8A8" in m_low.scheme and "W8A16" in m_high.scheme

def test_build_mixed_precision_no_target_overlap():
    m_low, m_high = build_mixed_precision_recipe(["down_proj"])
    sensitive_re = m_high.targets[0]
    assert sensitive_re.startswith("re:"), "敏感层 targets 应是 re: 正则"
    assert sensitive_re in m_low.ignore, "低比特组必须 ignore 掉敏感层（防 targets 重叠）"
    assert "lm_head" in m_low.ignore and "lm_head" in m_high.ignore, "lm_head 两组都 ignore"

def test_is_mixed_precision_true():
    qc = {"config_groups": {
        "group_0": {"targets": ["Linear"], "weights": {"num_bits": 8}},
        "group_1": {"targets": ["re:.*down_proj"], "weights": {"num_bits": 16}}}}
    assert is_mixed_precision(qc) is True

def test_is_mixed_precision_false_single_group():
    qc = {"config_groups": {"group_0": {"targets": ["Linear"], "weights": {"num_bits": 8}}}}
    assert is_mixed_precision(qc) is False, "单组不是 mixed"

def test_is_mixed_precision_false_same_bits():
    qc = {"config_groups": {
        "group_0": {"targets": ["Linear"], "weights": {"num_bits": 8}},
        "group_1": {"targets": ["re:.*x"], "weights": {"num_bits": 8}}}}  # 比特相同
    assert is_mixed_precision(qc) is False, "比特全相同不算 mixed"

# L1 必过守卫：ipytest.run 返回 pytest 退出码；非 0（有测试失败）→ 抛异常让 nbconvert 真挂。
# （用 ipytest.run() 而非 %%ipytest magic：magic 吞掉失败、exit_code 属性在本版不可靠。）
_ec = ipytest.run("-qq")
assert _ec == 0, f"L1 测试未全过（exit_code={_ec}），见上方 pytest 输出。"

## L2（tiny，CPU）：mixed-precision recipe 字段自洽

构造 mixed-precision recipe，验证两个 modifier 的 targets/scheme/ignore 自洽（不重叠、lm_head 都 ignore）。CPU 上不真跑 oneshot（要校准集+慢），只验 recipe 结构正确——结构对了，喂给 oneshot 自然出对 config_groups。


In [ ]:
## L2：tiny 模型 mixed-precision recipe 自洽性
tiny = Qwen2ForCausalLM(Qwen2Config(
    num_hidden_layers=2, hidden_size=64, intermediate_size=128,
    num_attention_heads=4, num_key_value_heads=2, vocab_size=320, tie_word_embeddings=False))
linear_names = [n for n, m in tiny.named_modules() if isinstance(m, nn.Linear)]

recipe = build_mixed_precision_recipe(["down_proj"], low_scheme="W8A8", high_scheme="W8A16")
m_low, m_high = recipe

# 模拟 llmcompressor 的 targets/ignore 命中，验证无重叠
import re as _re
def hits(targets, ignore, names):
    def name_match(name, t):
        if t == "Linear": return True
        if isinstance(t, str) and t.startswith("re:"): return bool(_re.search(t[3:], name))
        return name == t
    def ignored(name):
        for ig in ignore:
            if ig.startswith("re:"):
                if _re.search(ig[3:], name): return True
            elif name == ig: return True
        return False
    low_targets = targets if isinstance(targets, list) else [targets]
    return [n for n in names if any(name_match(n, t) for t in low_targets) and not ignored(n)]

low_hit  = hits(m_low.targets,  m_low.ignore,  linear_names)
high_hit = hits(m_high.targets, m_high.ignore, linear_names)
print(f"低比特 W8A8 组命中 {len(low_hit)} 层（应不含 down_proj/lm_head）")
print(f"高比特 W8A16 组命中 {len(high_hit)} 层（应全是 down_proj）")
print("高比特组示例：", high_hit[:2])

assert not (set(low_hit) & set(high_hit)), "两组 targets 不应重叠"
assert all("down_proj" in n for n in high_hit), "高比特组应只命中 down_proj"
assert all("down_proj" not in n for n in low_hit), "低比特组应排除 down_proj"
assert "lm_head" not in low_hit and "lm_head" not in high_hit, "lm_head 两组都不量化"
print("\nL2 通过：mixed-precision recipe 结构自洽，两组 targets 不重叠。")


## L3（H200，GPU 守卫）：真 7B mixed-precision 产物 config_groups

在 7B 上跑 mixed-precision（敏感层 W8A16、其余 W8A8），读产物 config.json 的 config_groups——确认 llmcompressor 真的生成了两组不同比特（用 is_mixed_precision 裁决）。


In [ ]:
import torch, os
def run_l3_mixed_precision():
    from datasets import load_dataset
    calib = load_dataset("wikitext", "wikitext-2-raw-v1", split="train").shuffle(seed=0)["text"][:64]
    recipe = build_mixed_precision_recipe(["down_proj"], low_scheme="W8A8", high_scheme="W8A16")
    out = OUT_ROOT / "s4_mixed_precision"
    oneshot(model=str(MODEL_DIR), tokenizer=str(MODEL_DIR), recipe=recipe,
            dataset=calib, num_calibration_samples=64, output_dir=str(out))
    import json
    qc = json.loads((out / "config.json").read_text())["quantization_config"]
    print("is_mixed_precision =", is_mixed_precision(qc))
    print(f"产物 @ {out}")

if torch.cuda.is_available() and not os.environ.get("SKIP_L3"):
    run_l3_mixed_precision()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（CPU/CI 只验 L1+L2 recipe 结构逻辑）")


## 产物检查


In [ ]:
import json
p = OUT_ROOT / "s4_mixed_precision" / "config.json"
if p.exists():
    qc = json.loads(p.read_text())["quantization_config"]
    groups = qc.get("config_groups", {})
    print("config_groups：")
    for gname, g in groups.items():
        w = g.get("weights", {} or {})
        a = g.get("input_activations")
        print(f"  {gname}: targets={g.get('targets')} weights_bits={w.get('num_bits')} act_bits={(a or {}).get('num_bits') if a else None}")
    print(f"\nis_mixed_precision = {is_mixed_precision(qc)}")
else:
    print(f"{p} 不存在（L3 未跑或被 SKIP_L3 跳过）")
